In [25]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "uher2008great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Uher-Call_2008_reversed-contingency_II_raw-data.sav")
complete_path_2 = os.path.join(original_data_pathway, "table1.csv")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [26]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)

# comp_out_path_df = os.path.join(original_data_pathway, 'Uher-Call_2008_reversed-contingency_II_raw-data.csv')
# df.to_csv(comp_out_path_df, encoding='utf-8-sig', index=False)

df['study_id']="uher2008great"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.columns
df.rename(columns={"subject": "ape",
    "species":"species_original",
    "exper":"experiment_name",
    "side":"chosen_side"}, inplace=True)


In [27]:
df['date']= pd.to_datetime(df['date'],format='%Y-%m-%d')
df['year']= df['date'].dt.year
df['month']= df['date'].dt.month
df['day']= df['date'].dt.day

In [28]:
comp_path_name_errors = os.path.join(pathway_gen, "uher2008great_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

table1 = pd.read_csv(complete_path_2)   
df= df.merge(table1,left_on='ape', right_on='name', how='left')


In [29]:
# df['experiment_name'].unique()
# code_list=["experiment_name"]
# for index, x in enumerate(code_list):    
#     df[x] = df[x].astype(str)
#     temp=[]
#     for entry in df[x]:
#         if entry == 'rev_02_04':
#             entry = "experienced"
#         elif entry =='rev_03_04':
#             entry = "naive"
#         temp.append(entry)
#     df = df.assign(temp_col=temp)
#     df=df.rename(columns={'temp_col': 'condition'})
# df = df.assign(experiment='1')

In [30]:
# df['receive_large'].replace('incorrect', 'no', inplace=True, regex=True)
# df['receive_large'].replace('correct', 'yes', inplace=True, regex=True)
df.rename(columns={"ape": "participant", 
                   'corr':'correct_choice',
                   'age':'age_in_years'}, inplace=True)

df.at[1605, 'chosen_side'] = 'missing'

In [31]:
df=df[['study_id', 'year', 'month', 'day',  'participant','age_in_years', 'sex', 'species',
     'session', 'trial', 'condition','left',
       'right', 'chosen_side', 'correct_choice' ]]

In [32]:
comp_out_path_stand = os.path.join(out_pathway, 'uher2008great_standardized.csv')
df.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names = df.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
uher2008great_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'uher2008great_glossary.csv')
uher2008great_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)